**Machine Learning Lab**

# 9. Neural Networks

**Neural networks** are a fascinating and complex aspect of machine learning, often compared to the human brain for their ability to learn from data. They consist of interconnected nodes or neurons that process information and make decisions. By adjusting weights and biases through a process known as training, neural networks can improve their accuracy over time, making them powerful tools for tasks like speech and image recognition. As they evolve, neural networks continue to push the boundaries of what machines can learn and accomplish.

Import pandas and numpy, which we'll use for loading and manipulating the dataset.

In [ ]:
import pandas as pd
import numpy as np

## Loading the Dataset

This notebook uses `diabetes.csv` (the Pima Indians Diabetes dataset), where each row is a patient described by 8 medical measurements, and `Outcome` indicates whether they were diagnosed with diabetes (1) or not (0). Run the cell below and upload `diabetes.csv` (found in this notebook's folder in the repo) when prompted.

In [ ]:
from google.colab import files
uploaded = files.upload()  # select diabetes.csv

Load the data and preview it.

In [ ]:
df = pd.read_csv("diabetes.csv")
df.head()

Check the size of the dataset.

In [ ]:
df.shape

In the next step, we divide the attributes into two parts: training features and label. Features (X) will be all the columns except **Outcome**, **Outcome** itself will be the label(Y).

In [ ]:
X = df.drop(['Outcome'], axis=1)
y = df['Outcome']
X.head()

In classification, data is usually divided into train and test sets. We will do the same here. For that purpose, Scikit-learn libraries train_test_split() function can be used. We will take 80% data for training, 20% data for testing. The split will be random. It should be noted that, there are other ways to split dataset, but we will use the percantage method.

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

In the last classes, we have tried different classifers like deision trees, support vector machine(SVM) and naive bayes. This time, we will use **artificial neural network (ANN)**. Artificial neural networks have an input layer, an output layer, and multiple hidden layers. The tensorflow and keras libraries can be used to create ANN.

In [ ]:
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, Dense

At first, we need to create the model. For this task, we will use a simple multilayer perceptron. **Next time you change the model, start executing code from here.**

In [ ]:
def create_model():
  # This line creates a blank sequential neural network
  model = Sequential()

  # Adds a hidden layer with 8 neurons
  # We are giving it 8 features, so input_dim=8
  # 'relu' will be our activation function. We could also use 'sigmoid', 'softmax', 'selu', etc.
  model.add(Input(shape=(8,)))
  model.add(Dense(2, activation='relu'))

  # Add more layers if you want
  model.add(Dense(4, activation='relu'))

  # The output could be contained in just one neuron, as it is basically 0 or 1
  # We will add another Dense layer for output with sigmoid activation
  model.add(Dense(1, activation='sigmoid'))

  # The model now needs to be compiled
  model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
  return model

Create and train the model. `validation_split=0.2` sets aside 20% of the *training* data to monitor performance during training (separate from the `X_test` we'll use for final evaluation).

In [ ]:
# Training step
model = create_model()
model.fit(X_train, y_train, epochs=100, batch_size=16, validation_split=0.2, verbose=1)

Our model is ready(yes, its THAT easy). But now, we need to check how accurate it is. To do that, we will first predict the label for the data stored in X_test. Then compare that prediction with the original label stored in Y_test.

In [ ]:
from sklearn.metrics import classification_report

y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten().tolist() # Convert probabilities to binary predictions

print(classification_report(y_test, y_pred))

Now, let's try to improve the performance. We will start with binning the data. `KBinsDiscretizer` groups the continuous `Age` column into 5 discrete buckets ('quantile' means each bucket gets roughly the same number of patients), turning age into more of a categorical signal for the network.

In [ ]:
from sklearn.preprocessing import KBinsDiscretizer

# Initialize KBinsDiscretizer
discretizer = KBinsDiscretizer(n_bins=5, encode='ordinal', strategy='quantile')

# Fit and transform the 'Age' column in X_train
X_train['Age_bin'] = discretizer.fit_transform(X_train[['Age']])
X_train = X_train.drop('Age', axis=1)

# Transform the 'Age' column in X_test using the same bins
X_test['Age_bin'] = discretizer.transform(X_test[['Age']])
X_test = X_test.drop('Age', axis=1)

X_train.head()

Retrain the same architecture from scratch on the binned data, and check whether the classification report improved.

In [ ]:
model = create_model()
model.fit(X_train, y_train, epochs=200, batch_size=16, validation_split=0.2, verbose=1)

y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten().tolist()
print(classification_report(y_test, y_pred))

Let's try by normalizing. `MinMaxScaler` rescales every feature into the [0, 1] range — neural networks generally train more smoothly when input features are on a similar, small scale, rather than having wildly different magnitudes (e.g. `Insulin` in the hundreds vs `DiabetesPedigreeFunction` under 3).

In [ ]:
# Normalize the data
from sklearn.preprocessing import MinMaxScaler

scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_train = pd.DataFrame(X_train, columns=X_test.columns)

X_test = scaler.transform(X_test)
X_test = pd.DataFrame(X_test, columns=X_train.columns)

X_train.head()

Retrain again on the normalized data.

In [ ]:
model = create_model()
model.fit(X_train, y_train, epochs=200, batch_size=16, validation_split=0.2, verbose=1)

y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten().tolist()
print(classification_report(y_test, y_pred))

One hot encoding. `Age_bin` currently holds integers 0–4, which implies a false order/magnitude relationship between bins. One-hot encoding it into 5 separate binary columns removes that false ordering. `.align()` afterward makes sure `X_train` and `X_test` end up with the exact same set of columns (in case one set happened to be missing a bin the other has).

In [ ]:
import pandas as pd
# Perform one-hot encoding on 'Age_bin'
X_train = pd.get_dummies(X_train, columns=['Age_bin'], prefix='Age')
X_test = pd.get_dummies(X_test, columns=['Age_bin'], prefix='Age')

# Align columns in X_train and X_test to handle potential missing columns after one-hot encoding
X_train, X_test = X_train.align(X_test, join='outer', axis=1, fill_value=0)

X_train.head()

Check how many columns we have now — the single `Age_bin` column has expanded into 5 one-hot columns.

In [ ]:
X_train.shape

As one hot encoding changes the shape of the dataset, we need to use another
ANN architecutre.

In [ ]:
def create_model_2():
  # This line creates a blank sequential neural network
  model = Sequential()

  # Adds a hidden layer with 8 neurons
  # We are giving it 8 features, so input_dim=8
  # 'relu' will be our activation function. We could also use 'sigmoid', 'softmax', 'selu', etc.
  model.add(Input(shape=(12,)))
  model.add(Dense(16, activation='relu'))

  # Add more layers if you want
  model.add(Dense(16, activation='relu'))
  model.add(Dense(16, activation='relu'))

  # The output could be contained in just one neuron, as it is basically 0 or 1
  # We will add another Dense layer for output with sigmoid activation
  model.add(Dense(1, activation='sigmoid'))

  # The model now needs to be compiled
  model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
  return model

We will also implement early stopping now. `EarlyStopping` watches the validation loss during training and stops once it stops improving for `patience` epochs in a row, then restores the weights from the best epoch — this avoids wasting time training past the point of overfitting.

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping

model = create_model_2()
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

model.fit(
    X_train, y_train,
    epochs=300,
    batch_size=16,
    validation_split=0.2,
    verbose=1,
    callbacks=[early_stopping]
)

y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten().tolist()
print(classification_report(y_test, y_pred))

Let's apply SMOTE to remove data imbalance. Recall `Outcome` has about 500 negative vs. 268 positive cases — SMOTE generates synthetic positive examples so the network sees a balanced dataset during training, rather than being biased toward always predicting the majority class.

In [ ]:
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

y_train_resampled.value_counts()

Retrain on the balanced (resampled) training data, and evaluate on the same, untouched `X_test`.

In [ ]:
model = create_model_2()
model.fit(X_train_resampled, y_train_resampled, epochs=100, batch_size=16, validation_split=0.2, verbose=1)

y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten().tolist()
print(classification_report(y_test, y_pred))

Designing neural networks is an art and science, both. Create your own architecture and see if you can go above 85%. This time, we will also use Dropout layers to tackle overfitting. `Dropout(0.5)` randomly "turns off" 50% of neurons in that layer during each training step, which forces the network not to rely too heavily on any single neuron and tends to reduce overfitting.

In [ ]:
from tensorflow.keras.layers import Dropout

def create_model_2():
  # This line creates a blank sequential neural network
  model = Sequential()

  # Adds a hidden layer with 8 neurons
  # We are giving it 8 features, so input_dim=8
  # 'relu' will be our activation function. We could also use 'sigmoid', 'softmax', 'selu', etc.
  model.add(Input(shape=(12,)))
  model.add(Dense(8, activation='relu'))
  # model.add(Dropout(0.5))

  # Add more layers if you want
  model.add(Dense(128, activation='relu'))
  model.add(Dropout(0.5))
  model.add(Dense(128, activation='relu'))
  model.add(Dropout(0.5))
  # model.add(Dense(250, activation='relu'))

  # The output could be contained in just one neuron, as it is basically 0 or 1
  # We will add another Dense layer for output with sigmoid activation
  model.add(Dense(1, activation='sigmoid'))

  # The model now needs to be compiled
  model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
  return model

model = create_model_2()
early_stopping = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

model.fit(
    X_train_resampled, y_train_resampled,
    epochs=200,
    batch_size=16,
    validation_split=0.2,
    verbose=0,
    callbacks=[early_stopping]
)

y_pred_prob = model.predict(X_test)
y_pred = (y_pred_prob > 0.5).astype(int).flatten().tolist()

print(classification_report(y_test, y_pred))

## Practice Problems

1. Uncomment the two commented-out `Dropout(0.5)`/`Dense(250, ...)` lines in the last architecture and re-run. Does adding more dropout/capacity help or hurt accuracy on this dataset?
2. Try changing the `patience` parameter of `EarlyStopping` (e.g. to 2 or to 15). How does that change how many epochs training actually runs for, and the final classification report?
3. Add a `class_weight` argument to `model.fit()` (a dict like `{0: 1, 1: 1.9}`, roughly matching the original 500:268 imbalance) instead of using SMOTE, and compare the results to the SMOTE-resampled version above. Which approach gives better recall on the positive class?
4. Try replacing every `'relu'` activation in `create_model_2` with `'tanh'`. Does it train faster or slower, and does final accuracy change?
5. The dataset has some biologically implausible zero values (e.g. `BloodPressure == 0`, `BMI == 0`), which likely represent missing data rather than true zeros. Try replacing zeros in `Glucose`, `BloodPressure`, `SkinThickness`, `Insulin`, and `BMI` with the column mean (excluding the zeros) before training. Does data cleaning like this improve the classification report?